# Analisis Runtun Waktu Kualitas Udara (Sentinel-5P) 
**Studi Kasus:** Kabupaten Bangkalan, Madura (Agustus 2025 - Agustus 2026)

## 1. Business Understanding
**Tujuan Proyek:** 
Memetakan dan memantau secara proaktif fluktuasi kualitas udara di wilayah Kabupaten Bangkalan menggunakan data penginderaan jauh satelit Sentinel-5P selama kurun waktu satu tahun terakhir.

**Manfaat Proyek:**
1. **Sistem Peringatan Dini Kesehatan:** Memberikan peringatan berbasis data bagi masyarakat dan kelompok rentan saat angka polutan harian melonjak agar membatasi aktivitas *outdoor*.
2. **Evaluasi Tata Ruang & Kebijakan:** Menjadi landasan objektif bagi instansi terkait maupun pengelola kawasan (seperti Universitas Trunojoyo Madura) untuk mengevaluasi kepadatan lalu lintas dan efektivitas ruang terbuka hijau.
3. **Transparansi Informasi Publik:** Menerjemahkan matriks data satelit yang rumit menjadi grafik *time series* visual di portal web statis agar ancaman "polusi tak kasatmata" mudah dipahami oleh warga sipil.

---
## 2. Data Understanding
Dataset ini berisi rekaman harian konsentrasi gas polutan yang diekstrak menggunakan agregasi spasial (*Polygon GeoJSON*) wilayah Bangkalan. 

**Deskripsi Fitur Polutan:**
* **CO (Karbon Monoksida):** *Silent killer* tak berbau dari pembakaran tidak sempurna, utamanya disumbang oleh asap knalpot kendaraan atau pembakaran sampah terbuka.
* **NO₂ (Nitrogen Dioksida):** Gas reaktif pemicu radang pernapasan, berasal dari emisi mesin bahan bakar fosil (terutama diesel) dan aktivitas industri.
* **O₃ (Ozon Permukaan):** Polutan sekunder berbahaya yang terbentuk dari reaksi kimia gas buang kendaraan di bawah terik matahari.
* **SO₂ (Sulfur Dioksida):** Gas berbau menyengat penyebab hujan asam, umumnya dari pembakaran batu bara atau emisi bahan bakar kapal laut di area pesisir.

**Catatan Eksplorasi Data (Anomali):**
Dalam data satelit optik, terdapat kondisi wajar yang perlu diperhatikan:
1. **Missing Values (NaN):** Terjadi ketika wilayah tertutup awan mendung/hujan, sehingga sensor satelit tidak dapat menembus permukaan bumi.
2. **Outliers (Pencilan):** Lonjakan nilai ekstrem pada hari tertentu yang bisa merekam insiden nyata (kebakaran lahan) atau sekadar *noise* instrumen.

---
## 3. Persiapan Lingkungan & Koneksi Server
Langkah teknis dimulai dengan memasang pustaka `openeo` dan mengautentikasi sistem lokal ke peladen Copernicus Data Space Ecosystem (CDSE). Koordinat Bangkalan didefinisikan menggunakan GeoJSON.

In [1]:
!pip install openeo
import openeo
import json
import pandas as pd
from functools import reduce
import time

print("Menghubungkan ke CDSE Copernicus...")
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

^C


  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached deprecated-1.3.1-py2.py3-none-any.whl.metadata (5.9 kB)
  Using cached oschmod-0.3.12-py2.py3-none-any.whl.metadata (10.0 kB)
     ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
     ---------------------------------------- 46.7/46.7 kB 2.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/349.2 kB ? eta -:--:--
   ---------------- ----------------------- 143.4/349.2 kB 4.3 MB/s eta 0:00:01
   -------------------------------------- - 337.9/349.2 kB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 349.2/349.2 kB 2.4 MB/s eta 0:00:00
Using cached deprecated-1.3.1-py2.py3-none-any.whl (11 kB)
   ---------------------------------------- 0.0/343.3 kB ? eta -:--:--
   --------------------------- ----------- 245.8/343.3 kB 15.7 MB/s eta 0:00:01
   ---------------------------------------- 343.3/343.3 